# 🎯 Classifier Showdown: Churn Prediction
**Author:** Mehr Ali | **Kaggle:** mehralieng | **GitHub:** mehrali-hub

---
Comparative analysis of 5 classifiers on a 440K-row telecom churn dataset with hyperparameter tuning, full evaluation, and business recommendation.


## 📦 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)

PALETTE = ['#6C63FF','#FF6584','#43D9AD','#FFB347','#4FC3F7']
plt.rcParams.update({'figure.dpi':120,'font.family':'DejaVu Sans'})
print('✅ Setup complete')

## 📂 2. Data Loading & Exploration

In [ ]:
# Load the 20K sample (included in /data folder)
df = pd.read_csv('data/churn_sample_20k.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Class Balance ===')
print(df['Churn'].value_counts(normalize=True).round(3))

In [ ]:
df.describe().round(2)

## 📊 3. Exploratory Data Analysis

In [ ]:
# Churn Distribution
fig, ax = plt.subplots(figsize=(6,4))
counts = df['Churn'].value_counts()
ax.bar(['No Churn','Churn'], counts.values, color=[PALETTE[0],PALETTE[1]], width=0.5, edgecolor='white')
ax.set_title('Churn Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
for i,v in enumerate(counts.values):
    ax.text(i, v+100, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout(); plt.savefig('images/01_churn_distribution.png'); plt.show()

In [ ]:
# Correlation Heatmap
num_cols = ['Age','Tenure','Usage Frequency','Support Calls','Payment Delay','Total Spend','Last Interaction','Churn']
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            mask=np.triu(np.ones((len(num_cols),len(num_cols)),dtype=bool)), ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('images/02_correlation_heatmap.png'); plt.show()

In [ ]:
# Age Distribution by Churn
fig, ax = plt.subplots(figsize=(8,4))
for c,col,lbl in zip([0,1],PALETTE,['No Churn','Churn']):
    ax.hist(df[df['Churn']==c]['Age'], bins=30, color=col, alpha=0.7, label=lbl, edgecolor='white')
ax.set_title('Age Distribution by Churn', fontsize=14, fontweight='bold')
ax.set_xlabel('Age'); ax.set_ylabel('Count'); ax.legend()
plt.tight_layout(); plt.savefig('images/03_age_distribution.png'); plt.show()

In [ ]:
# Subscription Type vs Churn
fig, axes = plt.subplots(1,2, figsize=(13,4))
crosstab = pd.crosstab(df['Subscription Type'], df['Churn'], normalize='index')*100
crosstab.columns=['No Churn','Churn']
crosstab.plot(kind='bar', ax=axes[0], color=[PALETTE[0],PALETTE[1]], edgecolor='white', width=0.55)
axes[0].set_title('Churn Rate by Subscription Type', fontweight='bold'); axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
crosstab2 = pd.crosstab(df['Contract Length'], df['Churn'], normalize='index')*100
crosstab2.columns=['No Churn','Churn']
crosstab2.plot(kind='bar', ax=axes[1], color=[PALETTE[0],PALETTE[1]], edgecolor='white', width=0.55)
axes[1].set_title('Churn Rate by Contract Length', fontweight='bold'); axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout(); plt.savefig('images/04_subscription_churn.png'); plt.show()

In [ ]:
# Support Calls Boxplot
fig, ax = plt.subplots(figsize=(6,4))
sns.boxplot(x='Churn', y='Support Calls', data=df, palette=[PALETTE[0],PALETTE[1]], ax=ax)
ax.set_xticklabels(['No Churn','Churn'])
ax.set_title('Support Calls by Churn Status', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('images/05_support_calls_boxplot.png'); plt.show()

## ⚙️ 4. Preprocessing

In [ ]:
df_enc = df.copy()
le = LabelEncoder()
for col in ['Gender','Subscription Type','Contract Length']:
    df_enc[col] = le.fit_transform(df_enc[col])
df_enc.drop('CustomerID', axis=1, inplace=True)

X = df_enc.drop('Churn', axis=1)
y = df_enc['Churn'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 🤖 5. Train Five Classifiers (Cross-Validation)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42), True),
    'KNN':                 (KNeighborsClassifier(n_neighbors=7), True),
    'Decision Tree':       (DecisionTreeClassifier(max_depth=8,class_weight='balanced',random_state=42), False),
    'Random Forest':       (RandomForestClassifier(n_estimators=150,class_weight='balanced',random_state=42,n_jobs=-1), False),
    'SVM':                 (SVC(kernel='rbf',probability=True,class_weight='balanced',random_state=42), True),
}
cv_results = {}
for name,(m,ns) in models.items():
    Xtr = X_train_sc if ns else X_train.values
    scores = cross_val_score(m, Xtr, y_train, cv=cv, scoring='f1', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:25s}: F1 = {scores.mean():.4f} ± {scores.std():.4f}')

## 🔧 6. Hyperparameter Tuning (GridSearchCV)

In [ ]:
# Logistic Regression tuning
gs_lr = GridSearchCV(LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42),
                     {'C':[0.01,0.1,1,10]}, cv=3, scoring='f1', n_jobs=-1)
gs_lr.fit(X_train_sc, y_train)
print(f'Best LR params: {gs_lr.best_params_} → F1={gs_lr.best_score_:.4f}')

In [ ]:
# Random Forest tuning
gs_rf = GridSearchCV(RandomForestClassifier(class_weight='balanced',random_state=42,n_jobs=-1),
                     {'n_estimators':[100,200],'max_depth':[8,12,None]}, cv=3, scoring='f1', n_jobs=-1)
gs_rf.fit(X_train.values, y_train)
print(f'Best RF params: {gs_rf.best_params_} → F1={gs_rf.best_score_:.4f}')

## 📈 7. Final Evaluation on Test Set

In [ ]:
final_models = {
    'Logistic Regression': (gs_lr.best_estimator_, True),
    'KNN':                 (models['KNN'][0], True),
    'Decision Tree':       (models['Decision Tree'][0], False),
    'Random Forest':       (gs_rf.best_estimator_, False),
    'SVM':                 (models['SVM'][0], True),
}
results = {}
for name,(m,ns) in final_models.items():
    Xtr = X_train_sc if ns else X_train.values
    Xte = X_test_sc  if ns else X_test.values
    m.fit(Xtr, y_train)
    yp  = m.predict(Xte)
    ypr = m.predict_proba(Xte)[:,1]
    results[name] = {'Accuracy':accuracy_score(y_test,yp),'Precision':precision_score(y_test,yp),
                     'Recall':recall_score(y_test,yp),'F1':f1_score(y_test,yp),
                     'ROC-AUC':roc_auc_score(y_test,ypr),'y_pred':yp,'y_prob':ypr}
results_df = pd.DataFrame({k:{m:v for m,v in r.items() if m not in ['y_pred','y_prob']} for k,r in results.items()}).T.round(4)
results_df

## 📊 8. Visualizations: Comparison Charts

In [ ]:
# CV F1 Boxplot
fig, ax = plt.subplots(figsize=(9,5))
bp = ax.boxplot(list(cv_results.values()), patch_artist=True, widths=0.5)
for patch,col in zip(bp['boxes'],PALETTE): patch.set_facecolor(col); patch.set_alpha(0.8)
ax.set_xticklabels(list(cv_results.keys()), rotation=15, ha='right')
ax.set_ylabel('F1 Score'); ax.set_title('5-Fold CV F1 per Classifier', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('images/08_cv_f1_boxplot.png'); plt.show()

In [ ]:
# Model Comparison Bar
metrics=['Accuracy','Precision','Recall','F1','ROC-AUC']
fig,ax=plt.subplots(figsize=(13,5))
x=np.arange(len(metrics)); w=0.13
for i,(name,col) in enumerate(zip(results_df.index,PALETTE)):
    ax.bar(x+i*w,[results_df.loc[name,m] for m in metrics],w,label=name,color=col,edgecolor='white')
ax.set_xticks(x+w*2); ax.set_xticklabels(metrics)
ax.set_ylim(0.55,1.02); ax.legend(loc='lower right',fontsize=9)
ax.set_title('Classifier Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('images/07_model_comparison.png'); plt.show()

In [ ]:
# ROC Curves
fig,ax=plt.subplots(figsize=(8,6))
for (name,r),col in zip(results.items(),PALETTE):
    fpr,tpr,_=roc_curve(y_test,r['y_prob'])
    ax.plot(fpr,tpr,color=col,lw=2,label=f"{name} (AUC={r['ROC-AUC']:.3f})")
ax.plot([0,1],[0,1],'--',color='gray')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves', fontsize=13, fontweight='bold'); ax.legend(loc='lower right',fontsize=9)
plt.tight_layout(); plt.savefig('images/10_roc_curves.png'); plt.show()

In [ ]:
# Confusion Matrices
fig,axes=plt.subplots(2,3,figsize=(15,9))
axes=axes.flatten()
for idx,(name,r) in enumerate(results.items()):
    sns.heatmap(confusion_matrix(y_test,r['y_pred']),annot=True,fmt='d',cmap='Purples',ax=axes[idx],
                cbar=False,xticklabels=['No Churn','Churn'],yticklabels=['No Churn','Churn'])
    axes[idx].set_title(name,fontweight='bold')
axes[-1].set_visible(False)
fig.suptitle('Confusion Matrices', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.savefig('images/09_confusion_matrices.png'); plt.show()

In [ ]:
# Feature Importance
best_rf_model = gs_rf.best_estimator_
fi = pd.Series(best_rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)
fig,ax=plt.subplots(figsize=(8,5))
ax.barh(fi.index,fi.values,color='#6C63FF',edgecolor='white')
ax.set_title('Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout(); plt.savefig('images/11_feature_importance.png'); plt.show()

## 📋 9. Classification Report (Best Model)

In [ ]:
best = 'Random Forest'
print(f'Best Model: {best}\n')
print(classification_report(y_test, results[best]['y_pred'], target_names=['No Churn','Churn']))

## 💼 10. Business Recommendation

**Recommended Model: Random Forest**

The Random Forest classifier achieves the highest performance across all metrics — **F1 = 0.993, ROC-AUC = 0.9999** on the held-out test set — making it the clear choice for production deployment. In the churn context, **false negatives** (missed churners) are more costly than false positives: retaining a customer through a targeted offer is far cheaper than acquiring a new one. Random Forest's near-perfect recall (98.6%) means almost no at-risk customer slips through undetected. Its built-in feature importance also provides actionable business insight — `Support Calls` and `Payment Delay` emerge as the strongest churn predictors, directly informing proactive intervention strategies. Additionally, unlike SVM, Random Forest requires no feature scaling, is robust to outliers, and scales well to the full 440K-row dataset in production.
